# 🌿 Plant Disease Prediction using CNN and Transfer Learning
### End-to-End Deep Learning Pipeline with MobileNetV2

This notebook implements an end-to-end computer vision workflow for identifying plant diseases from leaf imagery using **MobileNetV2** transfer learning. It is built strictly around the existing workspace dataset and executes the following pipeline:

1. **Library Imports & Environment Configuration**
2. **Dataset Inspection & Verification**
3. **Class Distribution & Imbalance Analysis**
4. **Visualizing Sample Leaf Images**
5. **Data Preprocessing & Normalization**
6. **Stratified Train / Validation / Test Splitting**
7. **Data Augmentation Pipeline**
8. **MobileNetV2 Architecture & Feature Extractor**
9. **Stage 1 Training: Classification Head (Frozen Base)**
10. **Stage 2 Training: Fine-Tuning Upper Convolutional Layers**
11. **Plotting Accuracy Curves**
12. **Plotting Loss Curves**
13. **Model Evaluation on Unseen Test Set**
14. **Confusion Matrix Heatmap**
15. **Classification Report (Precision, Recall, F1)**
16. **Sample Test Image Predictions**
17. **Explainable AI with Grad-CAM**
18. **Saving Model Artifacts and Class Mapping**

## 1. Import Libraries

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to Python module path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from src.config import (
    DATASET_PATH, IMAGE_SIZE, BATCH_SIZE, INITIAL_EPOCHS, FINE_TUNE_EPOCHS,
    INITIAL_LEARNING_RATE, FINE_TUNE_LEARNING_RATE, RANDOM_SEED, MODEL_PATH,
    BEST_MODEL_PATH, CLASS_NAMES_PATH, RESULTS_DIR
)
from src.data_loader import scan_dataset, create_train_val_test_splits, build_tf_dataset
from src.preprocessing import get_data_augmentation_pipeline, load_and_preprocess_image
from src.utils import set_seed, format_class_name, save_class_names, load_class_names

set_seed(RANDOM_SEED)
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Dataset Path: {DATASET_PATH}")

## 2. Inspect Dataset

In [ ]:
filepaths, labels, class_names, class_counts = scan_dataset(DATASET_PATH)
num_classes = len(class_names)
total_images = len(filepaths)

print(f"Total Valid Images: {total_images:,}")
print(f"Total Classes Detected: {num_classes}")

## 3. Display Class Distribution

In [ ]:
df_counts = pd.DataFrame(list(class_counts.items()), columns=["Class", "Count"])
df_counts["Formatted_Name"] = df_counts["Class"].apply(format_class_name)
df_counts = df_counts.sort_values(by="Count", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_counts, y="Formatted_Name", x="Count", palette="viridis")
plt.title("Plant Disease Class Distribution", fontsize=14, fontweight="bold")
plt.xlabel("Number of Images", fontsize=12)
plt.ylabel("Class Name", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print("Class Imbalance Analysis:")
print(f"Max Class Count: {df_counts['Count'].max()} ({df_counts.iloc[0]['Formatted_Name']})")
print(f"Min Class Count: {df_counts['Count'].min()} ({df_counts.iloc[-1]['Formatted_Name']})")
print(f"Imbalance Ratio: {df_counts['Count'].max() / df_counts['Count'].min():.2f}:1")

## 4. Display Sample Images

In [ ]:
plt.figure(figsize=(15, 8))
sample_indices = []
for c_idx in range(min(num_classes, 15)):
    idx = np.where(labels == c_idx)[0][0]
    sample_indices.append(idx)

for i, idx in enumerate(sample_indices):
    plt.subplot(3, 5, i + 1)
    img = Image.open(filepaths[idx])
    plt.imshow(img)
    plt.title(format_class_name(class_names[labels[idx]]), fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 5 & 6. Preprocess & Split Dataset

In [ ]:
(train_files, train_labels), (val_files, val_labels), (test_files, test_labels) = create_train_val_test_splits(
    filepaths=filepaths,
    labels=labels,
    random_seed=RANDOM_SEED
)

print(f"Train samples:      {len(train_files):,} ({len(train_files)/total_images*100:.1f}%)")
print(f"Validation samples: {len(val_files):,} ({len(val_files)/total_images*100:.1f}%)")
print(f"Test samples:       {len(test_files):,} ({len(test_files)/total_images*100:.1f}%)")

# Compute balanced class weights
class_weights = compute_class_weight("balanced", classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {int(c): float(w) for c, w in zip(np.unique(train_labels), class_weights)}

train_ds = build_tf_dataset(train_files, train_labels, batch_size=BATCH_SIZE, is_training=True)
val_ds = build_tf_dataset(val_files, val_labels, batch_size=BATCH_SIZE, is_training=False)
test_ds = build_tf_dataset(test_files, test_labels, batch_size=BATCH_SIZE, is_training=False)

## 7. Apply Augmentation Demonstration

In [ ]:
aug_pipeline = get_data_augmentation_pipeline()

sample_img = Image.open(train_files[0]).resize(IMAGE_SIZE)
sample_array = tf.expand_dims(tf.constant(np.array(sample_img, dtype=np.float32)), 0)

plt.figure(figsize=(12, 4))
plt.subplot(1, 4, 1)
plt.imshow(sample_img)
plt.title("Original Leaf")
plt.axis("off")

for i in range(3):
    augmented = aug_pipeline(sample_array, training=True)[0].numpy().astype("uint8")
    plt.subplot(1, 4, i + 2)
    plt.imshow(augmented)
    plt.title(f"Augmentation {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 8. Build MobileNetV2 Transfer Learning Architecture

In [ ]:
from src.model import build_model, configure_fine_tuning

model, base_model = build_model(
    num_classes=num_classes,
    learning_rate=INITIAL_LEARNING_RATE
)
model.summary(line_length=90)

## 9. Train Classification Head (Stage 1)

In [ ]:
callbacks = [
    ModelCheckpoint(filepath=str(BEST_MODEL_PATH), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, verbose=1)
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## 10. Fine-Tune Upper Layers (Stage 2)

In [ ]:
model = configure_fine_tuning(
    model=model,
    base_model=base_model,
    unfreeze_layers=30,
    fine_tune_lr=FINE_TUNE_LEARNING_RATE
)

initial_epoch = len(history_stage1.epoch)
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=initial_epoch,
    epochs=initial_epoch + FINE_TUNE_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## 11 & 12. Plot Training History (Accuracy & Loss)

In [ ]:
from src.utils import plot_training_history

history_dict = {
    "accuracy": history_stage1.history["accuracy"] + history_stage2.history.get("accuracy", []),
    "val_accuracy": history_stage1.history["val_accuracy"] + history_stage2.history.get("val_accuracy", []),
    "loss": history_stage1.history["loss"] + history_stage2.history.get("loss", []),
    "val_loss": history_stage1.history["val_loss"] + history_stage2.history.get("val_loss", []),
}

plot_training_history(history_dict, save_path=RESULTS_DIR / "training_history.png")

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history_dict["accuracy"], label="Train Accuracy", linewidth=2)
plt.plot(history_dict["val_accuracy"], label="Val Accuracy", linewidth=2)
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_dict["loss"], label="Train Loss", linewidth=2)
plt.plot(history_dict["val_loss"], label="Val Loss", linewidth=2)
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 13, 14 & 15. Evaluation on Unseen Test Set

In [ ]:
from src.evaluate import evaluate_model

metrics = evaluate_model(
    model=model,
    test_ds=test_ds,
    y_true=test_labels,
    class_names=class_names
)

## 16. Test Predictions on Individual Images

In [ ]:
from src.predict import PlantDiseasePredictor

predictor = PlantDiseasePredictor(model_path=BEST_MODEL_PATH, class_names_path=CLASS_NAMES_PATH)

test_sample_idx = 0
test_image_path = test_files[test_sample_idx]
true_label_name = format_class_name(class_names[test_labels[test_sample_idx]])

result = predictor.predict(test_image_path)
print(f"Ground Truth: {true_label_name}")
print(f"Prediction:   {result['formatted_class']}")
print(f"Confidence:   {result['confidence_percentage']:.2f}%")
print("\nTop 3 Predictions:")
for p in result["top_predictions"]:
    print(f"  {p['rank']}. {p['formatted_name']}: {p['percentage']:.2f}%")

## 17. Explainable AI with Grad-CAM

In [ ]:
superimposed, heatmap, orig = predictor.generate_gradcam(test_image_path)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(orig)
plt.title(f"Original: {true_label_name}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(superimposed)
plt.title(f"Grad-CAM Overlay ({result['formatted_class']})")
plt.axis("off")

plt.tight_layout()
plt.show()

## 18. Save Model & Class Names

In [ ]:
model.save(str(MODEL_PATH))
save_class_names(class_names, CLASS_NAMES_PATH)
print(f"[INFO] Model saved to: {MODEL_PATH}")
print(f"[INFO] Class names saved to: {CLASS_NAMES_PATH}")